In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Dropout, BatchNormalization

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler

from sklearn.metrics import mean_absolute_error, root_mean_squared_log_error
from sklearn.ensemble import RandomForestRegressor

from xgboost import XGBRegressor

In [ ]:
train = pd.read_csv("train.csv")

In [ ]:
test = pd.read_csv("test.csv")

In [ ]:
res = pd.read_csv("sample_submission.csv")

In [ ]:
train.drop("id", axis=1, inplace=True)

In [ ]:
train.store_and_fwd_flag = (train.store_and_fwd_flag == "N").astype("int8")

In [ ]:
test.store_and_fwd_flag = (test.store_and_fwd_flag == "N").astype("int8")

In [ ]:
train = pd.get_dummies(train, columns=["vendor_id"], dtype="int8")

In [ ]:
test = pd.get_dummies(test, columns=["vendor_id"], dtype="int8")

In [ ]:
train["pickup_datetime"] = pd.to_datetime(train["pickup_datetime"])
train["dropoff_datetime"] = pd.to_datetime(train["dropoff_datetime"])

In [ ]:
test["pickup_datetime"] = pd.to_datetime(test["pickup_datetime"])
test["dropoff_datetime"] = pd.to_datetime(test["dropoff_datetime"])

In [ ]:
train["duration"] = (train["dropoff_datetime"] - train["pickup_datetime"]).dt.seconds

In [ ]:
train.drop(index=train[(train["trip_duration"] != train["duration"])].index, inplace=True)

In [ ]:
train.drop(columns=["dropoff_datetime", "duration"], inplace=True)

In [ ]:
train.pickup_latitude = train.pickup_latitude * np.pi / 180
train.pickup_longitude	 = train.pickup_longitude * np.pi / 180
train.dropoff_latitude = train.dropoff_latitude * np.pi / 180
train.dropoff_longitude	 = train.dropoff_longitude	 * np.pi / 180

In [ ]:
R = 6371

In [ ]:
train["dist_euclid"] = ((train.pickup_latitude - train.dropoff_latitude) ** 2 + (train.pickup_longitude - train.dropoff_longitude) ** 2) ** 0.5 * R

In [ ]:
train["dist_manh"] = (np.abs(train.pickup_latitude - train.dropoff_latitude) + np.abs(train.pickup_longitude - train.dropoff_longitude)) * R

In [ ]:
phi1 = train.pickup_latitude
phi2 = train.dropoff_latitude
l1 = train.pickup_longitude
l2 = train.dropoff_longitude
train["dist_hav"] = 2 * R * np.arcsin(np.sqrt( np.sin((phi2 - phi1) / 2) ** 2 + np.cos(phi1) * np.cos(phi2) * np.sin((l2 - l1) / 2) ** 2  ) ) 

In [ ]:
train.drop(columns=["pickup_longitude", "pickup_latitude", "dropoff_longitude", "dropoff_latitude"], inplace=True)

In [ ]:
train["month"] = train.pickup_datetime.dt.month
train["day"] = train.pickup_datetime.dt.day
train["hour"] = train.pickup_datetime.dt.hour
train["minute"] = train.pickup_datetime.dt.minute
train["weekday"] = train.pickup_datetime.dt.weekday

In [ ]:
train = pd.get_dummies(train, columns=["weekday"], dtype="int8")

In [ ]:
train.drop(columns=["pickup_datetime"], inplace=True)

In [ ]:
x = train.drop("trip_duration", axis=1)
y = train["trip_duration"]

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [ ]:
x_train

In [ ]:
ms = MinMaxScaler()
x_train = ms.fit_transform(x_train)
x_test = ms.transform(x_test)

In [ ]:
model = RandomForestRegressor(max_depth=3)
model.fit(x_train, y_train)
pred_train = model.predict(x_train)
pred_test = model.predict(x_test)
mae_train = mean_absolute_error(y_train, pred_train)
mae_test = mean_absolute_error(y_test, pred_test)

In [ ]:
# pred_train[pred_train<0] = 0
# pred_test[pred_test<0] = 0

try:
    rmsle_train = root_mean_squared_log_error(y_train, pred_train)
    rmsle_test = root_mean_squared_log_error(y_test, pred_test)
except ValueError:
    ...
else:
    print(f"RMSLE: train={rmsle_train:.4f}, test={rmsle_test:.4f}")
print(f"MAE: train={mae_train:.4f}, test={mae_test:.4f}")

In [ ]:
nn = tf.keras.Sequential([
    Input(shape=x_train.shape[1:]),
    Dense(128, activation="relu"),
    Dense(64, activation="relu"),
    Dense(1, activation="linear")
])
nn.compile(optimizer="adam", loss="mse", metrics=["mae"])
nn.summary()

In [ ]:
nn.fit(x_train, y_train, epochs=10, batch_size=128, validation_data=(x_test, y_test))

In [ ]:
pred_train = nn.predict(x_train)
pred_test = nn.predict(x_test)
mae_train = mean_absolute_error(y_train, pred_train)
mae_test = mean_absolute_error(y_test, pred_test)

In [ ]:
# pred_train[pred_train<0] = 0
# pred_test[pred_test<0] = 0

try:
    rmsle_train = root_mean_squared_log_error(y_train, pred_train)
    rmsle_test = root_mean_squared_log_error(y_test, pred_test)
except ValueError:
    ...
else:
    print(f"RMSLE: train={rmsle_train:.4f}, test={rmsle_test:.4f}")
print(f"MAE: train={mae_train:.4f}, test={mae_test:.4f}")

In [ ]:
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping

In [ ]:
y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

In [ ]:
model = tf.keras.Sequential([
    Input(shape=x_train.shape[1:]),
    
    Dense(256, activation="relu"),
    BatchNormalization(),
    Dropout(0.1),  # 0.2-ից սարքեցինք 0.1, որ սովորելն արագանա
    
    Dense(128, activation="relu"),
    BatchNormalization(),
    Dropout(0.1),
    
    Dense(64, activation="relu"),
    BatchNormalization(),
    
    Dense(32, activation="relu"),
    
    Dense(1, activation="linear") # Քանի որ Y-ը լոգարիթմված է, սա կանխատեսելու է լոգարիթմական արժեք
])

# 2. Օպտիմիզատոր (առանց weight_decay-ի՝ սխալներից խուսափելու համար)
optimizer = tf.keras.optimizers.Adam(learning_rate=0.005) # Սկզբի համար մի քիչ մեծացրել ենք քայլը

model.compile(
    optimizer=optimizer, 
    loss="mse",        # MSE-ն լոգարիթմված Y-ի վրա հենց RMSLE-ն է
    metrics=["mae"]
)

# 3. Խելացի մարզման հրամաններ (Callbacks)
callbacks = [
    # Եթե մոդելը չի լավանում, փոքրացրու քայլը (learning rate)
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-5, verbose=1),
]

# 4. Մարզում (Օգտագործում ենք լոգարիթմված Y-ը)
history = model.fit(
    x_train, y_train_log, 
    validation_data=(x_test, y_test_log), 
    epochs=30, 
    batch_size=1024, # 1.1M տողի համար մեծ batch-ը հիանալի է
    callbacks=callbacks
)

In [ ]:
model.summary()

In [ ]:
model.fit(x_train, y_train, epochs=30, batch_size=128, validation_data=(x_test, y_test))